<a href="https://colab.research.google.com/github/zacw-243L/Oracle-Engine-to-estimate-Unifrom-amount/blob/main/Oracle%20Engine%20to%20estimate%20Unifrom%20amount.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oracle Engine

## Part 1: Prepare the uniform data

This section prepares the uniform collection file for the forecasting system.
**Just click the ▶ Run button beside each step, from 1 to 4.** Wait for each step to finish before moving on.

**Before starting:** Place the sample CSV named **`PCG MPC Uniform Collection-XXXXXXXXXXXXXXXXXXXXXXXX.csv`** in **My Drive** on Google Drive.
You can leave the filename as it is.

**When finished:** The prepared file will appear in **My Drive** as **`Cleaned Data.csv`**. Your original file will remain unchanged.



In [5]:
# @title 1. Connect Google Drive { display-mode: "form", single-column: true }
# @markdown Click **▶ Run**, choose the Google account containing the **sample uniform collection file**, and approve access. Colab will then load the file from My Drive.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Internal settings: staff do not need to enter filenames or folder paths.
_drive_folder = Path('/content/drive/MyDrive')
_original_name = 'PCG MPC Uniform Collection-6ab61ef0a70268fced1f06aa.csv'
_source_on_drive = _drive_folder / _original_name
_local_folder = Path('/content/oracle_engine_sanitizer')
_local_folder.mkdir(parents=True, exist_ok=True)
_raw_local = _local_folder / 'raw_input.csv'

if not _source_on_drive.is_file():
    raise FileNotFoundError(
        'The sample collection file was not found in My Drive. '
        'Put PCG MPC Uniform Collection-6ab61ef0a70268fced1f06aa.csv '
        'directly in My Drive and run Step 1 again.'
    )

shutil.copyfile(_source_on_drive, _raw_local)
print('Step 1 complete. The sample collection file is ready.')
print('Please run Step 2.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1 complete. The sample collection file is ready.
Please run Step 2.


In [6]:
# @title 2. Remove personal details { display-mode: "form", single-column: true }
# @markdown Click **▶ Run** to create a working copy containing only uniform collection information. The original file will remain unchanged.
import csv
from pathlib import Path

if '_raw_local' not in globals() or not _raw_local.is_file():
    raise RuntimeError('Please run Step 1 first.')

# Only these columns are allowed to pass through.
_working_columns = [
    'Timestamp', 'Intake',
    'No.4 Shirt (Size, Qty)', 'No.4 Trousers (Size, Qty)',
    'PCG No.4 Utility Belt (Size, Qty)',
    'PCG No.4 Tactical Belt (Size, Qty)',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)',
    'Insignia: Deckhand (Qty)', 'No.4 ERT Badge (Qty)',
    'Duffle Bag (Qty)', 'Utility Bag (Qty)'
]

with _raw_local.open('r', encoding='utf-8-sig', newline='') as file:
    _raw_lines = list(csv.reader(file))

# FormSG exports can contain a short summary above the response header.
_header_index = next((i for i, row in enumerate(_raw_lines)
                      if 'Response ID' in row and 'Timestamp' in row and 'Intake' in row), None)
if _header_index is None:
    raise ValueError('The file does not have the expected FormSG response headings.')

_raw_columns = _raw_lines[_header_index]
if len(_raw_columns) != len(set(_raw_columns)):
    raise ValueError('Duplicate column headings were found in the source file.')
_missing = [col for col in _working_columns if col not in _raw_columns]
if _missing:
    raise ValueError('The collection file is missing expected uniform fields: ' + ', '.join(_missing))

_working_records = []
_excluded_count = 0
for _line_number, _values in enumerate(_raw_lines[_header_index + 1:], start=_header_index + 2):
    if not any(v.strip() for v in _values):
        continue
    if len(_values) != len(_raw_columns):
        raise ValueError(f'Row {_line_number} has an unexpected number of fields.')
    _response = dict(zip(_raw_columns, _values))
    if _response.get('Download Status', 'Success').strip().lower() != 'success':
        _excluded_count += 1
        continue

    # This personal-Drive demonstration intentionally refuses non-synthetic input.
    if not _response.get('Name', '').strip().startswith('SYNTHETIC PERSON '):
        raise ValueError(
            'This demonstration is restricted to the supplied fictional test dataset. '
            'Do not use real personnel records in personal Colab.'
        )

    _row = {col: _response[col].strip() for col in _working_columns}
    if not _row['Timestamp'] or not _row['Intake']:
        raise ValueError(f'The source file has a missing date or intake at row {_line_number}.')
    _working_records.append(_row)

if not _working_records:
    raise ValueError('No successful sample responses were found.')

# The intermediate copy stays inside this temporary Colab session.
_working_path = _local_folder / 'working_data.csv'
with _working_path.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=_working_columns)
    writer.writeheader()
    writer.writerows(_working_records)

print(f'Step 2 complete. {len(_working_records):,} sample responses prepared.')
if _excluded_count:
    print(f'{_excluded_count:,} responses with unsuccessful download status were skipped.')
print('Names, response IDs, ranks, departments and frontline labels were removed.')
print('Please run Step 3.')

Step 2 complete. 1,416 sample responses prepared.
Names, response IDs, ranks, departments and frontline labels were removed.
Please run Step 3.


In [7]:
# @title 3. Clean and save the uniform data { display-mode: "form", single-column: true }
# @markdown Click **▶ Run** to prepare the final file. It will be saved to **My Drive** as **Cleaned Data.csv**. Running this step again replaces the previous Cleaned Data.csv.
import csv
import re
import shutil
from datetime import datetime
from pathlib import Path

if '_working_path' not in globals() or not _working_path.is_file():
    raise RuntimeError('Please run Step 2 first.')

# The final 9 columns match the forecasting example exactly.
_clean_columns = [
    'Timestamp', 'Intake',
    'No.4 Shirt (Size, Qty)', 'No.4 Trousers (Size, Qty)',
    'PCG No.4 Utility Belt (Size, Qty)',
    'PCG No.4 Tactical Belt (Size, Qty)',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)',
    'Insignia: Deckhand (Qty)', 'No.4 ERT Badge (Qty)'
]

with _working_path.open('r', encoding='utf-8-sig', newline='') as file:
    _reader = csv.DictReader(file)
    if not _reader.fieldnames or any(col not in _reader.fieldnames for col in _clean_columns):
        raise ValueError('The working data is missing required uniform fields. Run Step 2 again.')
    _clean_records = [{col: row[col] for col in _clean_columns} for row in _reader]

# Catch accidental changes to the FormSG table layout before exporting.
_patterns = {
    'No.4 Shirt (Size, Qty)': r'No\.4 Shirt size (?:[1-9]|10);\d+',
    'No.4 Trousers (Size, Qty)': r'No\.4 Trouser size (?:[1-9]|1[0-4]);\d+',
    'PCG No.4 Utility Belt (Size, Qty)': r'Utility Belt size (?:S|M|L|XL);\d+',
    'PCG No.4 Tactical Belt (Size, Qty)': r'Tactical Belt size (?:S|M|L|XL);\d+',
    'Fire & Cut Resistant Tactical Gloves (Size, Qty)': r'Fire & Cut Resistant Tactical Gloves Size (?:S|M|L|XL);\d+'
}
for _n, _record in enumerate(_clean_records, start=2):
    if not _record['Intake'].isdigit():
        raise ValueError(f'Invalid intake number at data row {_n}.')
    try:
        datetime.strptime(_record['Timestamp'], '%d/%m/%Y %H:%M')
    except ValueError:
        raise ValueError(f'Invalid timestamp at data row {_n}.') from None
    for _column, _pattern in _patterns.items():
        if not re.fullmatch(_pattern, _record[_column]):
            raise ValueError(f'Unrecognised uniform size or quantity at data row {_n}, column {_column}.')
    for _column in _clean_columns[7:]:
        if not _record[_column].isdigit():
            raise ValueError(f'Invalid item quantity at data row {_n}, column {_column}.')

# Write and verify locally before copying the completed file back to Drive.
_clean_local = _local_folder / 'Cleaned Data.csv'
with _clean_local.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=_clean_columns)
    writer.writeheader()
    writer.writerows(_clean_records)

_clean_drive = _drive_folder / 'Cleaned Data.csv'
shutil.copyfile(_clean_local, _clean_drive)

print('Step 3 complete. Cleaned Data.csv has been saved to My Drive.')
print(f'{len(_clean_records):,} responses and 9 uniform-data columns are ready.')
print('Please run Step 4 to check the results.')

Step 3 complete. Cleaned Data.csv has been saved to My Drive.
1,416 responses and 9 uniform-data columns are ready.
Please run Step 4 to check the results.


In [8]:
# @title 4. Check the cleaned data { display-mode: "form", single-column: true }
# @markdown Click **▶ Run** to see a sample of the cleaned file and check the number of personnel in each intake.
import pandas as pd
from IPython.display import display

if '_clean_drive' not in globals() or not _clean_drive.is_file():
    raise RuntimeError('Please run Step 3 first.')

_clean_preview = pd.read_csv(_clean_drive, encoding='utf-8-sig')
print('Data preparation completed successfully.')
print(f'File saved in My Drive: {_clean_drive.name}')
print(f'Total responses: {len(_clean_preview):,}')
print(f'Columns retained: {len(_clean_preview.columns)}')
print('\nFirst 10 records:')
display(_clean_preview.head(10))
print('\nNumber of people in each intake:')
display(_clean_preview['Intake'].value_counts().sort_index().rename('People').to_frame())

Data preparation completed successfully.
File saved in My Drive: Cleaned Data.csv
Total responses: 1,416
Columns retained: 9

First 10 records:


,Timestamp,Intake,"No.4 Shirt (Size, Qty)","No.4 Trousers (Size, Qty)","PCG No.4 Utility Belt (Size, Qty)","PCG No.4 Tactical Belt (Size, Qty)","Fire & Cut Resistant Tactical Gloves (Size, Qty)",Insignia: Deckhand (Qty),No.4 ERT Badge (Qty)
0,11/2/2020 11:31,1,No.4 Shirt size 5;4,No.4 Trouser size 6;4,Utility Belt size M;1,Tactical Belt size M;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2
1,11/2/2020 11:50,1,No.4 Shirt size 5;4,No.4 Trouser size 7;4,Utility Belt size M;1,Tactical Belt size M;1,Fire & Cut Resistant Tactical Gloves Size L;1,4,2
2,11/2/2020 12:03,1,No.4 Shirt size 3;4,No.4 Trouser size 5;4,Utility Belt size S;1,Tactical Belt size S;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2
3,11/2/2020 12:14,1,No.4 Shirt size 5;4,No.4 Trouser size 7;4,Utility Belt size M;1,Tactical Belt size M;1,Fire & Cut Resistant Tactical Gloves Size L;1,4,2
4,11/2/2020 12:16,1,No.4 Shirt size 1;4,No.4 Trouser size 2;4,Utility Belt size S;1,Tactical Belt size S;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2
5,11/2/2020 12:17,1,No.4 Shirt size 5;4,No.4 Trouser size 7;4,Utility Belt size M;1,Tactical Belt size M;1,Fire & Cut Resistant Tactical Gloves Size L;1,4,2
6,11/2/2020 12:23,1,No.4 Shirt size 8;4,No.4 Trouser size 10;4,Utility Belt size L;1,Tactical Belt size L;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2
7,11/2/2020 12:23,1,No.4 Shirt size 3;4,No.4 Trouser size 3;4,Utility Belt size S;1,Tactical Belt size S;1,Fire & Cut Resistant Tactical Gloves Size S;1,4,2
8,11/2/2020 12:27,1,No.4 Shirt size 7;4,No.4 Trouser size 8;4,Utility Belt size M;1,Tactical Belt size M;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2
9,11/2/2020 12:30,1,No.4 Shirt size 8;4,No.4 Trouser size 10;4,Utility Belt size L;1,Tactical Belt size L;1,Fire & Cut Resistant Tactical Gloves Size M;1,4,2



Number of people in each intake:


,People
Intake,
1,56
2,56
3,36
4,54
5,52
6,53
7,38
8,53
9,60
